# Results

In [2]:
import torch
import timm
import tome
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import eval_cpu
from eval_gpu import evaluate
from SReT import SReT_T_distill
import PiT_ToMe
import SReT_ToMe

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Current GPU Device: {torch.cuda.current_device()}")

batch_size = 128
dataset_dir = '/media/datasets/imagenet/val'
dataset_transform = transforms.Compose([
    transforms.Resize(256), 
    transforms.CenterCrop(224), 
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
dataset = datasets.ImageFolder(dataset_dir, transform=dataset_transform)
dataset_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
print(f'Loaded {len(dataset)} ImageNet-1k Validation images')

CUDA available: True
GPU Device Name: NVIDIA GeForce RTX 4060 Ti
Current GPU Device: 0
Loaded 50000 ImageNet-1k Validation images


#### DeiT-Tiny-Distill 

In [2]:
model = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=True)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.40 %
Total Parameters:                           5.91 M
Theoretical FLOPs:                          2.17 G
Throughput (BS=128):            1803.92 images/sec
Throughput (BS=64):             1935.23 images/sec
Throughput (BS=32):             2021.49 images/sec
Throughput (BS=16):             2341.51 images/sec
Throughput (BS=1):               714.46 images/sec
Peak Activation Memory (BS=128):         227.20 MB
Peak Activation Memory (BS=64):          113.03 MB
Peak Activation Memory (BS=32):           56.44 MB
Peak Activation Memory (BS=16):           28.59 MB
Peak Activation Memory (BS=1):             1.76 MB



In [3]:
_ = eval_cpu.evaluate("deit")

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   7.09 ms
Throughput:                         141.05 img/sec



#### DeiT-Tiny-Distill + ToMe | Constant Reduction

In [3]:
model = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=True)
tome.patch.timm(model, prop_attn=True)
model = model.cuda().eval()
model.r = 10
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            73.47 %
Total Parameters:                           5.91 M
Theoretical FLOPs:                          1.51 G
Throughput (BS=128):            2443.51 images/sec
Throughput (BS=64):             2670.01 images/sec
Throughput (BS=32):             2724.28 images/sec
Throughput (BS=16):             2663.20 images/sec
Throughput (BS=1):               247.49 images/sec
Peak Activation Memory (BS=128):         232.62 MB
Peak Activation Memory (BS=64):          117.20 MB
Peak Activation Memory (BS=32):           58.93 MB
Peak Activation Memory (BS=16):           29.64 MB
Peak Activation Memory (BS=1):             1.82 MB



In [4]:
_ = eval_cpu.evaluate("deit+tome+c", r=10)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   7.35 ms
Throughput:                         136.11 img/sec



In [4]:
model = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=True)
tome.patch.timm(model, prop_attn=True)
model = model.cuda().eval()
model.r = 20
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            61.49 %
Total Parameters:                           5.91 M
Theoretical FLOPs:                          0.92 G
Throughput (BS=128):            4059.78 images/sec
Throughput (BS=64):             4286.81 images/sec
Throughput (BS=32):             4249.97 images/sec
Throughput (BS=16):             3731.79 images/sec
Throughput (BS=1):               247.17 images/sec
Peak Activation Memory (BS=128):         216.16 MB
Peak Activation Memory (BS=64):          110.27 MB
Peak Activation Memory (BS=32):           55.12 MB
Peak Activation Memory (BS=16):           27.62 MB
Peak Activation Memory (BS=1):             1.69 MB



In [5]:
_ = eval_cpu.evaluate("deit+tome+c", r=20)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   5.52 ms
Throughput:                         181.15 img/sec



#### PiT-Tiny-Distill

In [5]:
model = timm.create_model("pit_ti_distilled_224", pretrained=True)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.42 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          1.01 G
Throughput (BS=128):            1792.98 images/sec
Throughput (BS=64):             1843.60 images/sec
Throughput (BS=32):             1931.32 images/sec
Throughput (BS=16):             2063.63 images/sec
Throughput (BS=1):               683.69 images/sec
Peak Activation Memory (BS=128):        1203.36 MB
Peak Activation Memory (BS=64):          603.73 MB
Peak Activation Memory (BS=32):          302.11 MB
Peak Activation Memory (BS=16):          151.92 MB
Peak Activation Memory (BS=1):             9.40 MB



In [6]:
_ = eval_cpu.evaluate("pit")

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   8.62 ms
Throughput:                         116.01 img/sec



#### PiT-Tiny-Distill + ToMe | Constant Reduction

In [6]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="constant", constant_r=10)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            73.76 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.80 G
Throughput (BS=128):            1567.11 images/sec
Throughput (BS=64):             1616.28 images/sec
Throughput (BS=32):             1655.81 images/sec
Throughput (BS=16):             1653.78 images/sec
Throughput (BS=1):               218.13 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [7]:
_ = eval_cpu.evaluate("pit+tome+c", r=10)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  10.39 ms
Throughput:                          96.26 img/sec



In [ ]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="constant", constant_r=20)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            71.09 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.66 G
Throughput (BS=128):            1723.47 images/sec
Throughput (BS=64):             1784.13 images/sec
Throughput (BS=32):             1811.17 images/sec
Throughput (BS=16):             1759.97 images/sec
Throughput (BS=1):               218.20 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [8]:
_ = eval_cpu.evaluate("pit+tome+c", r=20)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   9.87 ms
Throughput:                         101.29 img/sec



#### PiT-Tiny-Distill + ToMe | Linear Reduction

In [8]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="linear", total_budget=100)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.11 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.89 G
Throughput (BS=128):            1552.81 images/sec
Throughput (BS=64):             1596.75 images/sec
Throughput (BS=32):             1643.44 images/sec
Throughput (BS=16):             1647.18 images/sec
Throughput (BS=1):               237.81 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [9]:
_ = eval_cpu.evaluate("pit+tome+l", total_tokens=100)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  10.31 ms
Throughput:                          96.95 img/sec



In [9]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="linear", total_budget=300)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            63.69 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.63 G
Throughput (BS=128):            1902.64 images/sec
Throughput (BS=64):             1959.43 images/sec
Throughput (BS=32):             1946.63 images/sec
Throughput (BS=16):             1870.12 images/sec
Throughput (BS=1):               237.40 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [10]:
_ = eval_cpu.evaluate("pit+tome+l", total_tokens=300)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   9.44 ms
Throughput:                         105.90 img/sec



#### PiT-Tiny-Distill + ToMe | Exponential Reduction

In [10]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="exponential", initial_r_ratio=0.25, alpha=0.0)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            72.03 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.80 G
Throughput (BS=128):            1907.67 images/sec
Throughput (BS=64):             2000.54 images/sec
Throughput (BS=32):             2071.30 images/sec
Throughput (BS=16):             2082.22 images/sec
Throughput (BS=1):               371.18 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [11]:
_ = eval_cpu.evaluate("pit+tome+e", r_ratio=0.25, alpha=0)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   7.95 ms
Throughput:                         125.84 img/sec



In [11]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="exponential", initial_r_ratio=0.40, alpha=0.2)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            62.63 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.63 G
Throughput (BS=128):            2334.27 images/sec
Throughput (BS=64):             2441.46 images/sec
Throughput (BS=32):             2450.69 images/sec
Throughput (BS=16):             2345.04 images/sec
Throughput (BS=1):               289.74 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [12]:
_ = eval_cpu.evaluate("pit+tome+e", r_ratio=0.40, alpha=0.2)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   7.33 ms
Throughput:                         136.44 img/sec



In [12]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="exponential", initial_r_ratio=0.10, alpha=0.6)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            73.90 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.88 G
Throughput (BS=128):            1610.08 images/sec
Throughput (BS=64):             1656.02 images/sec
Throughput (BS=32):             1714.40 images/sec
Throughput (BS=16):             1712.87 images/sec
Throughput (BS=1):               228.57 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [13]:
_ = eval_cpu.evaluate("pit+tome+e", r_ratio=0.1, alpha=0.6)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  10.20 ms
Throughput:                          98.06 img/sec



#### SReT-Tiny-Distill

In [13]:
model = SReT_T_distill(pretrained=False) # initialize
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            77.44 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.91 G
Throughput (BS=128):            1072.04 images/sec
Throughput (BS=64):             1169.09 images/sec
Throughput (BS=32):             1250.25 images/sec
Throughput (BS=16):             1302.04 images/sec
Throughput (BS=1):               220.10 images/sec
Peak Activation Memory (BS=128):         795.76 MB
Peak Activation Memory (BS=64):          397.88 MB
Peak Activation Memory (BS=32):          200.88 MB
Peak Activation Memory (BS=16):          100.44 MB
Peak Activation Memory (BS=1):             6.22 MB



In [14]:
_ = eval_cpu.evaluate("sret")

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  11.95 ms
Throughput:                          83.68 img/sec



#### SReT-Tiny-Distill + ToMe | Constant Reduction

In [14]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="constant", constant_r=10)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            70.96 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.32 G
Throughput (BS=128):            1172.65 images/sec
Throughput (BS=64):             1265.42 images/sec
Throughput (BS=32):             1314.68 images/sec
Throughput (BS=16):             1169.07 images/sec
Throughput (BS=1):               112.36 images/sec
Peak Activation Memory (BS=128):         770.04 MB
Peak Activation Memory (BS=64):          385.13 MB
Peak Activation Memory (BS=32):          192.70 MB
Peak Activation Memory (BS=16):           96.22 MB
Peak Activation Memory (BS=1):             6.01 MB



In [15]:
_ = eval_cpu.evaluate("sret+tome+c", r=10)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  13.00 ms
Throughput:                          76.93 img/sec



In [15]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="constant", constant_r=20)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            40.95 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.06 G
Throughput (BS=128):            1327.79 images/sec
Throughput (BS=64):             1425.87 images/sec
Throughput (BS=32):             1462.26 images/sec
Throughput (BS=16):             1170.98 images/sec
Throughput (BS=1):               111.98 images/sec
Peak Activation Memory (BS=128):         756.22 MB
Peak Activation Memory (BS=64):          380.08 MB
Peak Activation Memory (BS=32):          189.43 MB
Peak Activation Memory (BS=16):           96.02 MB
Peak Activation Memory (BS=1):             5.91 MB



In [16]:
_ = eval_cpu.evaluate("sret+tome+c", r=20)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  12.10 ms
Throughput:                          82.66 img/sec



#### SReT-Tiny-Distill + ToMe | Linear Reduction

In [16]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="linear", total_budget=100)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            76.58 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.68 G
Throughput (BS=128):            1049.59 images/sec
Throughput (BS=64):             1135.24 images/sec
Throughput (BS=32):             1195.24 images/sec
Throughput (BS=16):             1186.15 images/sec
Throughput (BS=1):               112.84 images/sec
Peak Activation Memory (BS=128):         770.04 MB
Peak Activation Memory (BS=64):          385.13 MB
Peak Activation Memory (BS=32):          192.70 MB
Peak Activation Memory (BS=16):           96.22 MB
Peak Activation Memory (BS=1):             6.01 MB



In [17]:
_ = eval_cpu.evaluate("sret+tome+l", total_tokens=100)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  14.20 ms
Throughput:                          70.43 img/sec



In [17]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="linear", total_budget=300)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            61.36 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.26 G
Throughput (BS=128):            1293.42 images/sec
Throughput (BS=64):             1388.91 images/sec
Throughput (BS=32):             1421.85 images/sec
Throughput (BS=16):             1187.00 images/sec
Throughput (BS=1):               112.98 images/sec
Peak Activation Memory (BS=128):         742.90 MB
Peak Activation Memory (BS=64):          371.51 MB
Peak Activation Memory (BS=32):          188.13 MB
Peak Activation Memory (BS=16):           92.85 MB
Peak Activation Memory (BS=1):             5.80 MB



In [18]:
_ = eval_cpu.evaluate("sret+tome+l", total_tokens=300)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  12.49 ms
Throughput:                          80.07 img/sec



#### SReT-Tiny-Distill + ToMe | Exponential Reduction

In [18]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="exponential", initial_r_ratio=0.25, alpha=0.0)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            75.90 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.49 G
Throughput (BS=128):            1366.12 images/sec
Throughput (BS=64):             1493.88 images/sec
Throughput (BS=32):             1572.56 images/sec
Throughput (BS=16):             1578.22 images/sec
Throughput (BS=1):               175.68 images/sec
Peak Activation Memory (BS=128):         489.38 MB
Peak Activation Memory (BS=64):          245.46 MB
Peak Activation Memory (BS=32):          122.71 MB
Peak Activation Memory (BS=16):           62.09 MB
Peak Activation Memory (BS=1):             3.90 MB



In [19]:
_ = eval_cpu.evaluate("sret+tome+e", r_ratio=0.25, alpha=0)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  11.10 ms
Throughput:                          90.12 img/sec



In [19]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="exponential", initial_r_ratio=0.40, alpha=0.2)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            69.82 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.13 G
Throughput (BS=128):            1880.97 images/sec
Throughput (BS=64):             2023.09 images/sec
Throughput (BS=32):             2069.55 images/sec
Throughput (BS=16):             1847.04 images/sec
Throughput (BS=1):               146.18 images/sec
Peak Activation Memory (BS=128):         391.51 MB
Peak Activation Memory (BS=64):          195.76 MB
Peak Activation Memory (BS=32):           99.88 MB
Peak Activation Memory (BS=16):           48.94 MB
Peak Activation Memory (BS=1):             3.90 MB



In [21]:
_ = eval_cpu.evaluate("sret+tome+e", r_ratio=0.40, alpha=0.2)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  10.52 ms
Throughput:                          95.08 img/sec



In [20]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="exponential", initial_r_ratio=0.1, alpha=0.6)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            76.39 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.61 G
Throughput (BS=128):            1171.73 images/sec
Throughput (BS=64):             1275.46 images/sec
Throughput (BS=32):             1343.98 images/sec
Throughput (BS=16):             1329.05 images/sec
Throughput (BS=1):               130.32 images/sec
Peak Activation Memory (BS=128):         665.63 MB
Peak Activation Memory (BS=64):          335.25 MB
Peak Activation Memory (BS=32):          167.06 MB
Peak Activation Memory (BS=16):           83.09 MB
Peak Activation Memory (BS=1):             5.19 MB



In [22]:
_ = eval_cpu.evaluate("sret+tome+e", r_ratio=0.10, alpha=0.6)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  13.18 ms
Throughput:                          75.89 img/sec

